<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/yolov8_%EC%9C%A0%ED%8A%9C%EB%B8%8C%EB%8B%A4%EC%9A%B4%EB%A1%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics opencv-python-headless yt-dlp matplotlib

김영빈

In [ ]:
def play_and_save_youtube_with_yolo(youtube_url, skip_frames=5, output_filename="output_with_yolo.mp4", max_duration=10):
    """
    max_duration: 최대 처리 시간(초)
    """
    ydl_opts = {
        'format': 'mp4/best[height<=480]',
        'outtmpl': tempfile.gettempdir() + '/temp_video.%(ext)s',
        'quiet': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print("⬇️ 유튜브 영상 다운로드 중...")
            info = ydl.extract_info(youtube_url, download=True)
            video_path = ydl.prepare_filename(info)

            print(f"🎥 영상 제목: {info['title']}")

            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            max_frames = int(fps * max_duration)
            print(f"⏱ 최대 {max_duration}초 (약 {max_frames} 프레임) 처리")

            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_filename, fourcc, fps / skip_frames, (frame_width, frame_height))

            frame_num = 0
            processed_frames = 0

            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_num >= max_frames:
                    break

                if frame_num % skip_frames == 0:
                    results = model.predict(frame, imgsz=640, verbose=False)[0]
                    annotated_frame = results.plot()
                    out.write(annotated_frame)

                    clear_output(wait=True)
                    plt.figure(figsize=(10, 6))
                    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
                    plt.title(f"YOLOv8 추론 프레임 ({frame_num})")
                    plt.axis('off')
                    plt.show()

                    time.sleep(0.1)

                frame_num += 1

            cap.release()
            out.release()
            os.remove(video_path)
            print(f"✅ 영상 추론 종료, 저장 파일: {output_filename}")

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# 실행
youtube_url = input("유튜브 영상 URL 입력: ")
play_and_save_youtube_with_yolo(youtube_url, skip_frames=5, max_duration=10)


양근영

In [ ]:
!pip install yt-dlp

# 원하는 유튜브 영상 다운로드
!yt-dlp -f bestvideo+bestaudio --merge-output-format mp4 https://www.youtube.com/watch?v=tEtWnGwwCEc

# 다운로드된 mp4 파일 확인
for file in os.listdir("/content"): # /content 폴더에 있는 모든 파일 목록을 가져옴 (Colab 작업 디렉토리)
    if file.endswith(".mp4"): # 파일 이름이 .mp4로 끝나는지 확인 (즉, 동영상 파일인지 검사)
        print("📁 다운로드된 영상:", file)

# 모델 로드 (YOLOv8n - 경량 모델)
model = YOLO("yolov8n.pt")

# 영상 업로드
video_path = "강남대로 강남역 도로 드라이브 최근 밤거리 모습 입니다. [tEtWnGwwCEc].mp4"

# 저장 경로 및 비디오 캡처 설정
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 결과 저장 경로
output_path = "result_video_youtube.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# 프레임별로 추론 및 저장
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv8 추론
    results = model(frame, verbose=False)

    # 결과 이미지 (bounding box 포함)
    annotated_frame = results[0].plot()
    out.write(annotated_frame)

cap.release()
out.release()

# 결과 영상 표시
Video(output_path, embed=True)

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics opencv-python-headless yt-dlp

# 필요한 라이브러리 import
from ultralytics import YOLO
import cv2
import yt_dlp
import os
from IPython.display import HTML, Video
from base64 import b64encode

# 브라우저 쿠키를 사용한 YouTube 다운로드
def download_youtube_with_cookies(url, output_path="downloaded_video.%(ext)s"):
    """브라우저 쿠키를 사용해 YouTube 영상 다운로드"""

    # 여러 방법 시도
    methods = [
        {
            'name': 'Chrome 쿠키',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'cookiesfrombrowser': ('chrome', None, None, None),
            }
        },
        {
            'name': 'Firefox 쿠키',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'cookiesfrombrowser': ('firefox', None, None, None),
            }
        },
        {
            'name': '기본 방법',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
                }
            }
        }
    ]

    for method in methods:
        print(f"🔄 {method['name']} 시도 중...")
        try:
            with yt_dlp.YoutubeDL(method['opts']) as ydl:
                info = ydl.extract_info(url, download=True)
                filename = ydl.prepare_filename(info)

                # 실제 다운로드된 파일 찾기
                if os.path.exists(filename):
                    return filename

                # 확장자가 다를 수 있으므로 검색
                base_name = filename.rsplit('.', 1)[0]
                for ext in ['.mp4', '.webm', '.mkv']:
                    test_file = base_name + ext
                    if os.path.exists(test_file):
                        return test_file

        except Exception as e:
            print(f"❌ {method['name']} 실패: {e}")
            continue

    return None

# YouTube 영상 다운로드
print("📺 YouTube 영상 다운로드 시작...")
youtube_url = "https://www.youtube.com/watch?v=tEtWnGwwCEc"
video_path = download_youtube_with_cookies(youtube_url, "강남대로_영상.%(ext)s")

if not video_path:
    print("❌ YouTube 다운로드 실패")
    print("🔧 해결 방법:")
    print("1. 브라우저에서 YouTube에 로그인")
    print("2. VPN 사용")
    print("3. 다른 영상 URL 시도")
else:
    print(f"✅ 다운로드 성공: {video_path}")

    # 다운로드된 mp4 파일 확인
    print("\n📁 다운로드된 파일 목록:")
    for file in os.listdir("/content"):
        if file.endswith((".mp4", ".webm", ".mkv")):
            print(f"  - {file}")

    # YOLO 모델 로드
    print("\n🤖 YOLO 모델 로딩...")
    model = YOLO("yolov8n.pt")
    print("✅ 모델 로딩 완료")

    # 비디오 처리
    print(f"\n🎬 비디오 처리 시작: {video_path}")
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ 비디오 파일을 열 수 없습니다: {video_path}")
    else:
        # 비디오 정보
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"📹 해상도: {width}x{height}")
        print(f"📊 FPS: {fps:.1f}")
        print(f"⏱ 총 프레임: {frame_count}")

        # 결과 저장 설정
        output_path = "result_video_yolo.mp4"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        # 처리할 프레임 수 제한 (20초만)
        max_frames = min(int(fps * 20), frame_count)
        print(f"🎯 처리할 프레임: {max_frames} (약 20초)")

        # 프레임별 처리
        frame_num = 0
        print("\n🚀 YOLO 추론 시작...")

        while cap.isOpened() and frame_num < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            # YOLO 추론
            results = model(frame, verbose=False)

            # 결과 이미지 (bounding box 포함)
            annotated_frame = results[0].plot()
            out.write(annotated_frame)

            frame_num += 1

            # 진행률 표시 (100프레임마다)
            if frame_num % 100 == 0:
                progress = (frame_num / max_frames) * 100
                print(f"진행률: {progress:.1f}% ({frame_num}/{max_frames})")

        cap.release()
        out.release()

        print(f"\n✅ 처리 완료!")
        print(f"📁 결과 파일: {output_path}")

        # 파일 크기 확인
        if os.path.exists(output_path):
            file_size = os.path.getsize(output_path) / (1024 * 1024)
            print(f"📊 파일 크기: {file_size:.1f}MB")

            # 결과 영상 표시
            print("\n🎥 결과 영상:")
            try:
                # 방법 1: Video 함수 사용
                display(Video(output_path, embed=True, width=640, height=480))
            except:
                # 방법 2: HTML로 직접 표시
                try:
                    with open(output_path, 'rb') as f:
                        mp4_data = f.read()
                    data_url = f"data:video/mp4;base64,{b64encode(mp4_data).decode()}"

                    html_video = f"""
                    <video width="640" height="480" controls>
                        <source src="{data_url}" type="video/mp4">
                    </video>
                    """
                    display(HTML(html_video))
                except Exception as e:
                    print(f"❌ 비디오 표시 실패: {e}")
                    print(f"📁 파일 위치: {output_path}")
        else:
            print("❌ 결과 파일 생성 실패")

print("\n🎉 작업 완료!")

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics opencv-python-headless yt-dlp

# FFmpeg 설치 (동영상 인코딩에 필요)
!apt-get update
!apt-get install ffmpeg

from ultralytics import YOLO
import cv2
import yt_dlp
import os
import time
from IPython.display import HTML
from base64 import b64encode
import subprocess

def download_youtube_video(youtube_url, output_path='video.mp4'):
    """YouTube 영상 다운로드 - 브라우저 쿠키 사용"""

    # 3가지 방법 시도
    methods = [
        {
            'format': 'best[height<=720]',
            'outtmpl': output_path,
            'cookiesfrombrowser': ('chrome', None, None, None),
        },
        {
            'format': 'best[height<=720]',
            'outtmpl': output_path,
            'cookiesfrombrowser': ('firefox', None, None, None),
        },
        {
            'format': 'best[height<=720]',
            'outtmpl': output_path,
        }
    ]

    for i, ydl_opts in enumerate(methods):
        try:
            print(f"다운로드 방법 {i+1} 시도 중...")
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                print("동영상 다운로드 중...")
                ydl.download([youtube_url])

            if os.path.exists(output_path):
                print(f"✅ 다운로드 성공: {output_path}")
                return output_path

        except Exception as e:
            print(f"❌ 방법 {i+1} 실패: {str(e)}")
            continue

    print("❌ 모든 다운로드 방법 실패")
    return None

def process_video(model, video_path, output_path='output_temp.avi', skip_frames=2):
    """비디오에 대해 객체 탐지 수행"""
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise Exception("비디오 파일을 열 수 없습니다.")

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # AVI 형식으로 임시 저장 (XVID 코덱 사용)
        output_fps = fps // skip_frames
        print(f"비디오 정보: {width}x{height} @ {fps}fps → {output_fps}fps")

        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out = cv2.VideoWriter(output_path, fourcc, output_fps, (width, height))

        frame_count = 0
        processed_count = 0
        start_time = time.time()

        while cap.isOpened():
            success, frame = cap.read()
            frame_count += 1

            if not success:
                break

            if frame_count % skip_frames != 0:
                continue

            results = model.predict(frame, show=False)
            annotated_frame = results[0].plot()

            # BGR to RGB 변환
            annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            out.write(cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR))

            processed_count += 1

            if processed_count % 10 == 0:
                elapsed_time = time.time() - start_time
                progress = (frame_count / total_frames) * 100
                fps_processing = processed_count / elapsed_time
                remaining_frames = (total_frames - frame_count) // skip_frames
                eta = remaining_frames / fps_processing if fps_processing > 0 else 0

                print(f'진행률: {progress:.1f}% | 처리 속도: {fps_processing:.1f}fps | 남은 시간: {eta:.1f}초')

        cap.release()
        out.release()

        # AVI를 MP4로 변환 (FFmpeg 사용)
        final_output = 'output_final.mp4'
        print("\nMP4로 변환 중...")
        subprocess.run([
            'ffmpeg', '-i', output_path,
            '-c:v', 'libx264',
            '-preset', 'medium',
            '-crf', '23',
            '-c:a', 'aac',
            '-strict', 'experimental',
            final_output
        ])

        # 임시 파일 삭제
        os.remove(output_path)

        total_time = time.time() - start_time
        print(f"\n처리 완료!")
        print(f"총 소요시간: {total_time:.1f}초")

        return final_output

    except Exception as e:
        print(f"비디오 처리 중 에러 발생: {str(e)}")
        return None

def display_video_player(video_path):
    """비디오 플레이어 표시"""
    try:
        mp4 = open(video_path, 'rb').read()
        data_url = f"data:video/mp4;base64,{b64encode(mp4).decode()}"
        return HTML(f"""
        <video width="640" height="480" controls>
            <source src="{data_url}" type="video/mp4">
        </video>
        """)
    except Exception as e:
        print(f"비디오 표시 중 에러 발생: {str(e)}")
        return None

def main():
    try:
        youtube_url = 'https://www.youtube.com/watch?v=tEtWnGwwCEc'  # 원하는 URL로 변경

        print("YOLO 모델 로딩 중...")
        model = YOLO('yolov8n.pt')

        video_path = download_youtube_video(youtube_url)

        if video_path and os.path.exists(video_path):
            output_path = process_video(model, video_path)

            if output_path and os.path.exists(output_path):
                print("\n결과 영상을 재생합니다...")
                return display_video_player(output_path)
            else:
                print("비디오 처리 결과를 찾을 수 없습니다.")
        else:
            print("다운로드된 비디오 파일을 찾을 수 없습니다.")
            print("🔧 해결 방법:")
            print("1. 브라우저에서 YouTube에 로그인")
            print("2. VPN 사용")
            print("3. 다른 YouTube URL 시도")

    except Exception as e:
        print(f"실행 중 에러 발생: {str(e)}")

if __name__ == "__main__":
    display(main())# 필요한 패키지 설치
!pip install ultralytics opencv-python-headless yt-dlp

# 필요한 라이브러리 import
from ultralytics import YOLO
import cv2
import yt_dlp
import os
from IPython.display import HTML, Video
from base64 import b64encode

# 브라우저 쿠키를 사용한 YouTube 다운로드
def download_youtube_with_cookies(url, output_path="downloaded_video.%(ext)s"):
    """브라우저 쿠키를 사용해 YouTube 영상 다운로드"""

    # 여러 방법 시도
    methods = [
        {
            'name': 'Chrome 쿠키',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'cookiesfrombrowser': ('chrome', None, None, None),
            }
        },
        {
            'name': 'Firefox 쿠키',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'cookiesfrombrowser': ('firefox', None, None, None),
            }
        },
        {
            'name': '기본 방법',
            'opts': {
                'format': 'bestvideo+bestaudio/best',
                'merge_output_format': 'mp4',
                'outtmpl': output_path,
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
                }
            }
        }
    ]

    for method in methods:
        print(f"🔄 {method['name']} 시도 중...")
        try:
            with yt_dlp.YoutubeDL(method['opts']) as ydl:
                info = ydl.extract_info(url, download=True)
                filename = ydl.prepare_filename(info)

                # 실제 다운로드된 파일 찾기
                if os.path.exists(filename):
                    return filename

                # 확장자가 다를 수 있으므로 검색
                base_name = filename.rsplit('.', 1)[0]
                for ext in ['.mp4', '.webm', '.mkv']:
                    test_file = base_name + ext
                    if os.path.exists(test_file):
                        return test_file

        except Exception as e:
            print(f"❌ {method['name']} 실패: {e}")
            continue

    return None

# YouTube 영상 다운로드
print("📺 YouTube 영상 다운로드 시작...")
youtube_url = "https://www.youtube.com/watch?v=tEtWnGwwCEc"
video_path = download_youtube_with_cookies(youtube_url, "강남대로_영상.%(ext)s")

if not video_path:
    print("❌ YouTube 다운로드 실패")
    print("🔧 해결 방법:")
    print("1. 브라우저에서 YouTube에 로그인")
    print("2. VPN 사용")
    print("3. 다른 영상 URL 시도")
else:
    print(f"✅ 다운로드 성공: {video_path}")

    # 다운로드된 mp4 파일 확인
    print("\n📁 다운로드된 파일 목록:")
    for file in os.listdir("/content"):
        if file.endswith((".mp4", ".webm", ".mkv")):
            print(f"  - {file}")

    # YOLO 모델 로드
    print("\n🤖 YOLO 모델 로딩...")
    model = YOLO("yolov8n.pt")
    print("✅ 모델 로딩 완료")

    # 비디오 처리
    print(f"\n🎬 비디오 처리 시작: {video_path}")
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ 비디오 파일을 열 수 없습니다: {video_path}")
    else:
        # 비디오 정보
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"📹 해상도: {width}x{height}")
        print(f"📊 FPS: {fps:.1f}")
        print(f"⏱ 총 프레임: {frame_count}")

        # 결과 저장 설정
        output_path = "result_video_yolo.mp4"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        # 처리할 프레임 수 제한 (20초만)
        max_frames = min(int(fps * 20), frame_count)
        print(f"🎯 처리할 프레임: {max_frames} (약 20초)")

        # 프레임별 처리
        frame_num = 0
        print("\n🚀 YOLO 추론 시작...")

        while cap.isOpened() and frame_num < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            # YOLO 추론
            results = model(frame, verbose=False)

            # 결과 이미지 (bounding box 포함)
            annotated_frame = results[0].plot()
            out.write(annotated_frame)

            frame_num += 1

            # 진행률 표시 (100프레임마다)
            if frame_num % 100 == 0:
                progress = (frame_num / max_frames) * 100
                print(f"진행률: {progress:.1f}% ({frame_num}/{max_frames})")

        cap.release()
        out.release()

        print(f"\n✅ 처리 완료!")
        print(f"📁 결과 파일: {output_path}")

        # 파일 크기 확인
        if os.path.exists(output_path):
            file_size = os.path.getsize(output_path) / (1024 * 1024)
            print(f"📊 파일 크기: {file_size:.1f}MB")

            # 결과 영상 표시
            print("\n🎥 결과 영상:")
            try:
                # 방법 1: Video 함수 사용
                display(Video(output_path, embed=True, width=640, height=480))
            except:
                # 방법 2: HTML로 직접 표시
                try:
                    with open(output_path, 'rb') as f:
                        mp4_data = f.read()
                    data_url = f"data:video/mp4;base64,{b64encode(mp4_data).decode()}"

                    html_video = f"""
                    <video width="640" height="480" controls>
                        <source src="{data_url}" type="video/mp4">
                    </video>
                    """
                    display(HTML(html_video))
                except Exception as e:
                    print(f"❌ 비디오 표시 실패: {e}")
                    print(f"📁 파일 위치: {output_path}")
        else:
            print("❌ 결과 파일 생성 실패")

print("\n🎉 작업 완료!")

전은서

In [ ]:
!pip install ultralytics

from ultralytics import YOLO
from google.colab import files

# COCO 사전 훈련된 YOLOv8n 모델 로드
model = YOLO("yolov8n.pt")
model.info()  # 모델 정보 확인
# 동영상 파일 업로드
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
# 동영상 파일에 대해 YOLOv8n 모델로 추론 실행 및 결과 저장
results = model.predict(video_path, save=True)  # save=True로 결과 파일 저장

# 결과 영상 파일 이름 확인
import glob
import os

# 결과 파일은 'runs/detect/predict' 디렉터리에 저장됩니다.
result_dir = "runs/detect/predict"
output_files = glob.glob(os.path.join(result_dir, "*.mp4"))

# Colab에서 결과 영상 다운로드
from google.colab import files
for file in output_files:
    files.download(file)


김동건

In [ ]:
!pip install ultralytics yt-dlp opencv-python

import cv2
import os
from ultralytics import YOLO
from IPython.display import Video, display

def download_and_analyze_youtube_video(youtube_url):
    """
    YouTube 영상을 직접 다운로드하고 YOLO 분석
    """
    print("⬇️ YouTube 영상 다운로드 중...")

    # YouTube 영상 다운로드 (yt-dlp 직접 명령어 사용)
    os.system(f'yt-dlp -f "bestvideo+bestaudio/best[height<=720]" --merge-output-format mp4 "{youtube_url}"')

    # 다운로드된 mp4 파일 찾기
    video_file = None
    for file in os.listdir("/content"):
        if file.endswith(".mp4"):
            video_file = file
            print(f"📁 다운로드된 영상: {file}")
            break

    if not video_file:
        print("❌ 다운로드된 영상을 찾을 수 없습니다.")
        return None

    # YOLO 모델 로드
    print("🤖 YOLO 모델 로드 중...")
    model = YOLO("yolov8n.pt")

    # 비디오 캡처 설정
    print("🎬 영상 분석 시작...")
    video_path = f"/content/{video_file}"
    cap = cv2.VideoCapture(video_path)

    # 비디오 정보 가져오기
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps

    print(f"📹 영상 정보:")
    print(f"   해상도: {width}x{height}")
    print(f"   FPS: {fps:.1f}")
    print(f"   총 프레임: {total_frames}")
    print(f"   길이: {duration:.1f}초")

    # 결과 저장 설정
    output_path = "result_video_youtube.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # 분석 통계
    frame_count = 0
    detection_count = 0
    frames_with_objects = 0

    # 프레임별로 추론 및 저장
    print("🔍 YOLO 분석 진행 중...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # YOLOv8 추론
        results = model(frame, verbose=False, conf=0.5)

        # 탐지된 객체 수 계산
        if len(results) > 0 and results[0].boxes is not None:
            num_detections = len(results[0].boxes)
            detection_count += num_detections
            if num_detections > 0:
                frames_with_objects += 1

        # 결과 이미지 (bounding box 포함)
        annotated_frame = results[0].plot()
        out.write(annotated_frame)

        frame_count += 1

        # 진행률 표시 (10% 단위)
        if frame_count % (total_frames // 10) == 0:
            progress = (frame_count / total_frames) * 100
            print(f"진행률: {progress:.0f}%")

    # 리소스 해제
    cap.release()
    out.release()

    # 분석 결과 통계
    print("\n" + "="*50)
    print("📊 분석 완료! 결과 통계")
    print("="*50)
    print(f"총 처리 프레임: {frame_count}")
    print(f"총 탐지된 객체: {detection_count}")
    print(f"객체가 탐지된 프레임: {frames_with_objects}")
    if frame_count > 0:
        print(f"프레임당 평균 탐지 수: {detection_count/frame_count:.2f}")
        print(f"객체 탐지율: {(frames_with_objects/frame_count)*100:.1f}%")

    print(f"\n✅ 결과 영상 저장: {output_path}")

    # 결과 영상 표시
    print("🎥 결과 영상 재생:")
    return output_path

def analyze_local_video(video_path):
    """
    로컬 비디오 파일 YOLO 분석 (업로드된 파일용)
    """
    if not os.path.exists(video_path):
        print(f"❌ 파일을 찾을 수 없습니다: {video_path}")
        return None

    # YOLO 모델 로드
    print("🤖 YOLO 모델 로드 중...")
    model = YOLO("yolov8n.pt")

    # 비디오 캡처 설정
    print("🎬 영상 분석 시작...")
    cap = cv2.VideoCapture(video_path)

    # 비디오 정보 가져오기
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    print(f"📹 영상 정보:")
    print(f"   해상도: {width}x{height}")
    print(f"   FPS: {fps:.1f}")
    print(f"   총 프레임: {total_frames}")
    print(f"   길이: {duration:.1f}초")

    # 결과 저장 설정
    output_path = "result_video_local.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # 분석 통계
    frame_count = 0
    detection_count = 0
    frames_with_objects = 0

    # 프레임별로 추론 및 저장
    print("🔍 YOLO 분석 진행 중...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # YOLOv8 추론
        results = model(frame, verbose=False, conf=0.5)

        # 탐지된 객체 수 계산
        if len(results) > 0 and results[0].boxes is not None:
            num_detections = len(results[0].boxes)
            detection_count += num_detections
            if num_detections > 0:
                frames_with_objects += 1

        # 결과 이미지 (bounding box 포함)
        annotated_frame = results[0].plot()
        out.write(annotated_frame)

        frame_count += 1

        # 진행률 표시 (10% 단위)
        if total_frames > 0 and frame_count % max(1, total_frames // 10) == 0:
            progress = (frame_count / total_frames) * 100
            print(f"진행률: {progress:.0f}%")

    # 리소스 해제
    cap.release()
    out.release()

    # 분석 결과 통계
    print("\n" + "="*50)
    print("📊 분석 완료! 결과 통계")
    print("="*50)
    print(f"총 처리 프레임: {frame_count}")
    print(f"총 탐지된 객체: {detection_count}")
    print(f"객체가 탐지된 프레임: {frames_with_objects}")
    if frame_count > 0:
        print(f"프레임당 평균 탐지 수: {detection_count/frame_count:.2f}")
        print(f"객체 탐지율: {(frames_with_objects/frame_count)*100:.1f}%")

    print(f"\n✅ 결과 영상 저장: {output_path}")

    return output_path

# 메인 실행 부분
print("🎬 YouTube YOLO 분석기 (간단 버전)")
print("="*50)

mode = input("\n선택하세요:\n1. YouTube URL 입력\n2. 파일 업로드\n\n선택 (1 또는 2): ")

if mode == "1":
    # YouTube URL 입력
    youtube_url = input("\nYouTube URL을 입력하세요: ")

    result_path = download_and_analyze_youtube_video(youtube_url)

    if result_path and os.path.exists(result_path):
        # 결과 영상 표시
        display(Video(result_path, embed=True, width=800))
    else:
        print("❌ 분석 실패")

elif mode == "2":
    # 파일 업로드
    print("\n📁 비디오 파일을 업로드하세요:")
    try:
        from google.colab import files
        uploaded = files.upload()

        if uploaded:
            video_file = list(uploaded.keys())[0]
            print(f"📹 업로드된 파일: {video_file}")

            result_path = analyze_local_video(video_file)

            if result_path and os.path.exists(result_path):
                # 결과 영상 표시
                display(Video(result_path, embed=True, width=800))
            else:
                print("❌ 분석 실패")
        else:
            print("❌ 파일이 업로드되지 않았습니다.")

    except ImportError:
        # Colab이 아닌 환경
        video_path = input("비디오 파일 경로를 입력하세요: ")
        result_path = analyze_local_video(video_path)

        if result_path and os.path.exists(result_path):
            print(f"✅ 결과 영상: {result_path}")
        else:
            print("❌ 분석 실패")

else:
    print("❌ 잘못된 선택입니다.")

print("\n🎉 작업 완료!")